In [1]:
import torch
import triton
from triton_flash_attn import triton_flash_attn_varlen_func
from flash_attn import flash_attn_varlen_func
import random
from copy import deepcopy

import os
os.environ['TRITON_PRINT_AUTOTUNING'] = '1'

def compare(x, y, prefix=""):
    if x is None or y is None:
        return
    if any([x.dtype == torch.float32, y.dtype==torch.float32]):
        x,y = x.float(), y.float()
    diff = (x-y).abs()
    if prefix:
        print(prefix, end=": ")
    print(f"最大差异: {diff.max().item()}, 平均差异: {diff.mean().item()}")

def custom_flash_attn_varlen_func(q, k, v, cu_seqlens, max_len, causal=True, window_size=-1):
    d = q.size(-1)
    vd = v.size(-1)
    if d != vd:
        v = torch.nn.functional.pad(v, (0, q.size(-1) - v.size(-1)))
    o = flash_attn_varlen_func(q, k, v, cu_seqlens, cu_seqlens, max_len, max_len, causal=causal, window_size=window_size, return_attn_probs=False)
    if d != vd:
        o = o[..., :vd].contiguous()
    return o

print(triton.__version__)
# 建议最新版，3.2肯定不行

3.4.0


In [8]:
device = "cuda"
dtype = torch.bfloat16

def alloc_fn(size: int, align: int, _):
    return torch.empty(size, dtype=torch.int8, device=device)
triton.set_allocator(alloc_fn)

b, qh, kh, d = 4, 32, 8, 128
off = 0
splits = [random.randint(4096, 8192) for _ in range(b)]
splits = [1024 * 8] * 2
t = sum(splits)
cu_seqlens = torch.tensor([0] + splits, device=device).cumsum(0).to(torch.int32)
max_len = max(splits)

q1 = torch.randn(t, qh, d+off, device=device, dtype=dtype).requires_grad_(True)
k1 = torch.randn(t, kh, d+off, device=device, dtype=dtype).requires_grad_(True)
v1 = torch.randn(t, kh, d, device=device, dtype=dtype).requires_grad_(True)
q2, k2, v2 = deepcopy(q1), deepcopy(k1), deepcopy(v1)
torch.cuda.empty_cache()

In [12]:
window_size = (512, 512)
causal = False
o1, lse = triton_flash_attn_varlen_func(q1, k1, v1, cu_seqlens, max_len, causal=causal, window_size=window_size)
o1
o2 = custom_flash_attn_varlen_func(q2, k2, v2, cu_seqlens, max_len, causal=causal, window_size=window_size)
compare(o1, o2)
# compare(lse.transpose(0, 1), lse2 * 1.44)

do = torch.randn_like(o1)
o1.backward(do)
o2.backward(do)
compare(q1.grad, q2.grad)
compare(k1.grad, k2.grad)
compare(v1.grad, v2.grad)

最大差异: 0.001953125, 平均差异: 8.392333984375e-05
最大差异: 0.015625, 平均差异: 8.392333984375e-05
最大差异: 0.03125, 平均差异: 0.0004138946533203125
最大差异: 0.0625, 平均差异: 0.0003814697265625


In [14]:
print(triton.testing.do_bench(lambda: triton_flash_attn_varlen_func(q1, k1, v1, cu_seqlens, max_len, causal=True, window_size=window_size)))
print(triton.testing.do_bench(lambda: custom_flash_attn_varlen_func(q2, k2, v2, cu_seqlens, max_len, causal=True, window_size=window_size)))
o1, lse = triton_flash_attn_varlen_func(q1, k1, v1, cu_seqlens, max_len, causal=True, window_size=window_size)
o2 = custom_flash_attn_varlen_func(q2, k2, v2, cu_seqlens, max_len, causal=True, window_size=window_size)
do = torch.randn_like(o1)

print(triton.testing.do_bench(lambda: o1.backward(do, retain_graph=True), grad_to_none=[q1, k1, v1]))
print(triton.testing.do_bench(lambda: o2.backward(do, retain_graph=True), grad_to_none=[q2, k2, v2]))


2.1952410262563955
3.135111718342222
7.9357920090357466
9.306631088256836


In [5]:
q_idx = torch.arange(8)
k_idx = torch.arange(8)
w = 2

In [7]:
mask = (q_idx[:, None] >= k_idx[None, :]) & (q_idx[:, None] - w < k_idx[None, :])
mask

tensor([[ True, False, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False, False],
        [False,  True,  True, False, False, False, False, False],
        [False, False,  True,  True, False, False, False, False],
        [False, False, False,  True,  True, False, False, False],
        [False, False, False, False,  True,  True, False, False],
        [False, False, False, False, False,  True,  True, False],
        [False, False, False, False, False, False,  True,  True]])